# PCFI261 - Semana 10

## Redes densas para señales físicas y astronómicas

Este notebook acompaña las clases de la Semana 10. La idea es dar un paso más allá del caso Pima Indians Diabetes sin cambiar todavía a arquitecturas más complejas como CNN.

Trabajaremos con dos problemas autocontenidos:

1. **Clase 1:** una red densa como emulador de un oscilador amortiguado.
2. **Clase 2:** una red densa para detectar tránsitos de exoplanetas en curvas de luz sintéticas.

Ambos ejemplos usan datos simulados, por lo que no se requiere descargar archivos externos.

## Preparación

En Colab conviene activar GPU desde `Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU`. Para problemas pequeños la diferencia puede ser modesta; para los experimentos de escala debería notarse más.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("Dispositivos disponibles:")
for device in tf.config.list_physical_devices():
    print(" -", device.name, device.device_type)

plt.rcParams["figure.figsize"] = (7, 3.5)
plt.rcParams["axes.grid"] = True

# Clase 1 - Emulador de un oscilador amortiguado

Un oscilador amortiguado simple puede escribirse como

\[
x(t)=A e^{-\gamma t}\cos(\omega t + \phi).
\]

Entrenaremos una red densa que recibe `t`, `A`, `gamma`, `omega` y `phi`, y predice la posición `x(t)`.

Esto no reemplaza la física. Es un ejemplo de **emulador**: una aproximación rápida entrenada a partir de un modelo conocido.

In [ ]:
osc_rng = np.random.default_rng(42)

def oscillator(t, A, gamma, omega, phi):
    return A * np.exp(-gamma * t) * np.cos(omega * t + phi)

n_curves = 600
n_times = 80
osc_t_grid = np.linspace(0.0, 8.0, n_times)

rows = []
for _ in range(n_curves):
    A = osc_rng.uniform(0.5, 2.0)
    gamma = osc_rng.uniform(0.03, 0.35)
    omega = osc_rng.uniform(1.0, 4.0)
    phi = osc_rng.uniform(0.0, np.pi)
    x = oscillator(osc_t_grid, A, gamma, omega, phi)
    x += osc_rng.normal(0.0, 0.015, size=n_times)

    for t, xt in zip(osc_t_grid, x):
        rows.append([t, A, gamma, omega, phi, xt])

osc_data = pd.DataFrame(rows, columns=["t", "A", "gamma", "omega", "phi", "x"])
osc_X = osc_data[["t", "A", "gamma", "omega", "phi"]].to_numpy()
osc_y = osc_data["x"].to_numpy()

osc_X_train, osc_X_tmp, osc_y_train, osc_y_tmp = train_test_split(
    osc_X, osc_y, test_size=0.30, random_state=42
)
osc_X_val, osc_X_test, osc_y_val, osc_y_test = train_test_split(
    osc_X_tmp, osc_y_tmp, test_size=0.50, random_state=42
)

osc_scaler = StandardScaler()
osc_X_train_s = osc_scaler.fit_transform(osc_X_train)
osc_X_val_s = osc_scaler.transform(osc_X_val)
osc_X_test_s = osc_scaler.transform(osc_X_test)

osc_data.head()

In [ ]:
# Visualizamos algunas curvas generadas por el modelo físico.
plt.figure(figsize=(8, 4))
for i, (A, gamma, omega, phi) in enumerate([
    (1.8, 0.08, 1.6, 0.2),
    (1.2, 0.22, 2.8, 1.0),
    (0.8, 0.12, 3.6, 2.0),
]):
    plt.plot(osc_t_grid, oscillator(osc_t_grid, A, gamma, omega, phi), label=f"curva {i+1}")
plt.xlabel("t")
plt.ylabel("x(t)")
plt.title("Osciladores amortiguados sintéticos")
plt.legend()
plt.show()

## Entrenar la red de regresión

A diferencia del problema de diabetes, aquí no usamos `sigmoid` al final ni `binary_crossentropy`. La salida es una cantidad continua, por lo que usamos una salida lineal y error cuadrático medio.

In [ ]:
tf.keras.utils.set_random_seed(42)

osc_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(5,)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(1),
])

osc_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse",
    metrics=[tf.keras.metrics.RootMeanSquaredError(name="rmse")],
)

osc_history = osc_model.fit(
    osc_X_train_s, osc_y_train,
    validation_data=(osc_X_val_s, osc_y_val),
    epochs=40,
    batch_size=256,
    verbose=0,
)

osc_test_loss, osc_test_rmse = osc_model.evaluate(osc_X_test_s, osc_y_test, verbose=0)
print(f"MSE test:  {osc_test_loss:.5f}")
print(f"RMSE test: {osc_test_rmse:.5f}")

In [ ]:
plt.figure(figsize=(7, 3.5))
plt.plot(osc_history.history["loss"], label="train")
plt.plot(osc_history.history["val_loss"], label="validación")
plt.xlabel("época")
plt.ylabel("MSE")
plt.title("Curva de aprendizaje del emulador")
plt.legend()
plt.show()

In [ ]:
# Evaluamos una curva completa no usada explícitamente para entrenar.
case = {"A": 1.35, "gamma": 0.12, "omega": 2.7, "phi": 0.4}
osc_t_plot = np.linspace(0.0, 8.0, 250)

osc_X_curve = np.column_stack([
    osc_t_plot,
    np.full_like(osc_t_plot, case["A"]),
    np.full_like(osc_t_plot, case["gamma"]),
    np.full_like(osc_t_plot, case["omega"]),
    np.full_like(osc_t_plot, case["phi"]),
])

osc_y_true = oscillator(osc_t_plot, **case)
osc_y_pred = osc_model.predict(osc_scaler.transform(osc_X_curve), verbose=0).ravel()
osc_rmse_curve = np.sqrt(np.mean((osc_y_true - osc_y_pred) ** 2))

plt.figure(figsize=(8, 4))
plt.plot(osc_t_plot, osc_y_true, label="modelo físico", lw=2)
plt.plot(osc_t_plot, osc_y_pred, "--", label="red densa", lw=2)
plt.xlabel("t")
plt.ylabel("x(t)")
plt.title(f"Curva no vista: RMSE={osc_rmse_curve:.4f}")
plt.legend()
plt.show()

## Experimento opcional: escala y GPU

Para que una GPU se note, necesitamos bastante trabajo matricial. Repetimos el dataset para crear una carga más grande y medimos el tiempo de entrenamiento.

In [ ]:
def build_osc_regressor(width=256):
    net = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(5,)),
        tf.keras.layers.Dense(width, activation="relu"),
        tf.keras.layers.Dense(width, activation="relu"),
        tf.keras.layers.Dense(width, activation="relu"),
        tf.keras.layers.Dense(1),
    ])
    net.compile(optimizer="adam", loss="mse")
    return net

factor = 8
osc_X_big = np.tile(osc_X_train_s, (factor, 1))
osc_y_big = np.tile(osc_y_train, factor)

big_osc_model = build_osc_regressor(width=256)
t0 = time.perf_counter()
big_osc_model.fit(osc_X_big, osc_y_big, epochs=8, batch_size=1024, verbose=0)
osc_elapsed = time.perf_counter() - t0

print(f"Ejemplos usados: {len(osc_X_big):,}")
print(f"Tiempo entrenamiento: {osc_elapsed:.2f} s")

# Clase 2 - Detección de tránsitos de exoplanetas

Ahora pasamos a una aplicación astronómica. Simularemos curvas de luz con y sin tránsito planetario. Cada curva es un vector de flujo medido en distintos tiempos.

Seguimos usando redes densas: no hay convoluciones todavía.

In [ ]:
transit_rng = np.random.default_rng(123)

def make_light_curves(n_samples=6000, n_points=128, noise=0.003):
    time_grid = np.linspace(-1.0, 1.0, n_points)
    curves = np.empty((n_samples, n_points), dtype="float32")
    labels = np.empty(n_samples, dtype="int32")

    for i in range(n_samples):
        has_transit = transit_rng.random() < 0.5
        flux = 1.0 + transit_rng.normal(0.0, noise, size=n_points)

        if has_transit:
            depth = transit_rng.uniform(0.008, 0.030)
            width = transit_rng.uniform(0.06, 0.16)
            center = transit_rng.uniform(-0.25, 0.25)
            in_transit = np.abs(time_grid - center) < width
            flux[in_transit] -= depth

        curves[i] = flux
        labels[i] = int(has_transit)

    return time_grid, curves, labels

transit_time_grid, transit_curves, transit_labels = make_light_curves()

tr_X_train, tr_X_tmp, tr_y_train, tr_y_tmp = train_test_split(
    transit_curves, transit_labels, test_size=0.30, stratify=transit_labels, random_state=123
)
tr_X_val, tr_X_test, tr_y_val, tr_y_test = train_test_split(
    tr_X_tmp, tr_y_tmp, test_size=0.50, stratify=tr_y_tmp, random_state=123
)

transit_scaler = StandardScaler()
tr_X_train_s = transit_scaler.fit_transform(tr_X_train)
tr_X_val_s = transit_scaler.transform(tr_X_val)
tr_X_test_s = transit_scaler.transform(tr_X_test)

print("Curvas:", transit_curves.shape)
print("Fracción con tránsito:", transit_labels.mean().round(3))

In [ ]:
# Miramos ejemplos con y sin tránsito.
fig, axes = plt.subplots(2, 3, figsize=(11, 5), sharex=True, sharey=True)
example_idx = np.r_[np.where(transit_labels == 0)[0][:3], np.where(transit_labels == 1)[0][:3]]

for ax, idx in zip(axes.ravel(), example_idx):
    ax.plot(transit_time_grid, transit_curves[idx], lw=1.5)
    ax.set_title("con tránsito" if transit_labels[idx] == 1 else "sin tránsito")
    ax.set_xlabel("tiempo relativo")
    ax.set_ylabel("flujo")
plt.tight_layout()
plt.show()

## Entrenar el clasificador

La salida vuelve a ser una probabilidad, igual que en Pima. Usamos `sigmoid` y `binary_crossentropy`.

In [ ]:
tf.keras.utils.set_random_seed(123)

transit_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(tr_X_train_s.shape[1],)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

transit_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
)

transit_history = transit_model.fit(
    tr_X_train_s, tr_y_train,
    validation_data=(tr_X_val_s, tr_y_val),
    epochs=25,
    batch_size=256,
    verbose=0,
)

results = transit_model.evaluate(tr_X_test_s, tr_y_test, verbose=0)
print(dict(zip(transit_model.metrics_names, results)))

In [ ]:
plt.figure(figsize=(8, 3.5))
plt.plot(transit_history.history["accuracy"], label="accuracy train")
plt.plot(transit_history.history["val_accuracy"], label="accuracy validación")
plt.xlabel("época")
plt.ylabel("accuracy")
plt.title("Entrenamiento del clasificador de tránsitos")
plt.legend()
plt.show()

In [ ]:
transit_prob = transit_model.predict(tr_X_test_s, verbose=0).ravel()
transit_pred = (transit_prob >= 0.5).astype(int)

print(confusion_matrix(tr_y_test, transit_pred))
print(classification_report(tr_y_test, transit_pred, target_names=["sin tránsito", "con tránsito"]))

In [ ]:
# Casos cercanos al umbral: son interesantes para discutir incertidumbre.
idx = np.argsort(np.abs(transit_prob - 0.5))[:4]
fig, axes = plt.subplots(2, 2, figsize=(8, 5), sharex=True, sharey=True)
for ax, j in zip(axes.ravel(), idx):
    ax.plot(transit_time_grid, tr_X_test[j], lw=1.5)
    ax.set_title(f"real={tr_y_test[j]}, p={transit_prob[j]:.2f}")
    ax.set_xlabel("tiempo relativo")
    ax.set_ylabel("flujo")
plt.tight_layout()
plt.show()

## Actividad breve

Modifica un parámetro a la vez y observa qué pasa con la validación:

- subir el ruido;
- bajar la profundidad del tránsito;
- mover más el centro del tránsito;
- cambiar el número de puntos temporales;
- comparar una red pequeña contra una red más ancha.

## Experimento opcional: clasificación a mayor escala

La siguiente celda genera más curvas sintéticas. En GPU debería ser más interesante que el ejemplo pequeño, aunque sigue siendo un problema docente.

In [ ]:
def build_large_transit_classifier(n_points):
    net = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(n_points,)),
        tf.keras.layers.Dense(512, activation="relu"),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])
    net.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return net

_, tr_X_big_raw, tr_y_big = make_light_curves(n_samples=40000, n_points=128, noise=0.004)
tr_X_big = transit_scaler.transform(tr_X_big_raw)

big_transit_model = build_large_transit_classifier(tr_X_big.shape[1])
t0 = time.perf_counter()
big_transit_model.fit(tr_X_big, tr_y_big, epochs=6, batch_size=1024, validation_split=0.2, verbose=0)
transit_elapsed = time.perf_counter() - t0

print(f"Ejemplos usados: {len(tr_X_big):,}")
print(f"Tiempo entrenamiento: {transit_elapsed:.2f} s")

# Cierre

En esta semana usamos redes densas para dos tareas científicas distintas:

- **Regresión:** aproximar una función física parametrizada.
- **Clasificación:** detectar una señal astronómica sintética.

La arquitectura no cambió demasiado. Lo que cambió fue la pregunta científica, la interpretación de la salida y las métricas.